#Installing Libraries

In [ ]:
# @title
!pip install ultralytics
!pip install opencv-python-headless
!pip install networkx
%pip install -q "huggingface_hub>=0.25.0,<2.0"
!pip install pytesseract

!git clone https://github.com/boomb0om/CRAFT-text-detection.git
!git clone https://github.com/madmaze/pytesseract.git

import os
import sys
sys.path.append('/content/CRAFT-text-detection')

import yaml
import cv2
import torch
import numpy as np
import networkx as nx
import pytesseract
import re
import huggingface_hub
import gradio as gr

from ultralytics import YOLO
from matplotlib import pyplot as plt
from collections import defaultdict
from CRAFT import CRAFTModel, draw_polygons
from PIL import Image
from shapely.geometry import Point, Polygon, LineString
from huggingface_hub import get_token, whoami

print("-----All libraries installed-----")

#YOLO Symbol Detection

In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/Kia_Sportage_Labeled_Images.zip -d /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/images /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/labels /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/classes.txt /content/custom_data

!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.8

# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

# !yolo detect train data=/content/data.yaml model=yolo11m.pt epochs=60 imgsz=640

# Line Detection

In [ ]:
import time

# Load the model created above
# model = YOLO("/content/runs/detect/train-2/weights/best.pt")
model = YOLO("/content/best.pt")

# Obtain the class names from classes.txt
with open("/content/custom_data/classes.txt", "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

# Can specify classes to ignore for analysis
ignored_classes = {""}

# Margin of tolerance between endpoints of a line and a bounding box for a symbol
margin = 25
threshold = 50
minLineLength = 0
maxLineGap = 20

# Change the look of images
showSymbolLabel = True
showLineLabel = False

# Check if a point is near a bounding box for symbols
def is_point_near_box(point, box, margin=30):
    px, py = point
    x1, y1, x2, y2 = box
    return (x1 - margin <= px <= x2 + margin) and (y1 - margin <= py <= y2 + margin)

# Check if a line intersects a bounding box for symbols
def does_line_intersect_box(p1, p2, box):
    x1, y1, x2, y2 = box
    rect_lines = [((x1, y1), (x2, y1)), ((x2, y1), (x2, y2)),
                  ((x2, y2), (x1, y2)), ((x1, y2), (x1, y1))]
    return any(do_lines_intersect(p1, p2, rp1, rp2) for rp1, rp2 in rect_lines)

# Setup CRAFT
craft_model = CRAFTModel('weights/', 'cpu', use_refiner=False, fp16=False)

# Use tesseract to extract text from CRAFT polygons
def recognize_text_from_polygons(image, polygons):
    texts = []
    for poly in polygons:
        pts = np.array(poly, dtype=np.int32)
        x, y, w, h = cv2.boundingRect(pts)
        roi = image[y:y+h, x:x+w]
        text = pytesseract.image_to_string(roi, config="--psm 7").strip()
        if text:
            texts.append((poly, text))
    return texts

def assign_symbol_ids_with_text(symbol_boxes, recognized_text, symbol_classes, class_names, margin=25):
    symbol_ids = []
    label_count = defaultdict(int)

    for idx, box in enumerate(symbol_boxes):
        x1, y1, x2, y2 = box
        class_name = class_names[symbol_classes[idx]]
        assigned_text = None

        # Check nearby OCR text
        for poly, text in recognized_text:
            px, py, pw, ph = cv2.boundingRect(np.array(poly))
            if (x1 - margin <= px + pw/2 <= x2 + margin) and (y1 - margin <= py + ph/2 <= y2 + margin):
                valid_text = validate_symbol_name(text, class_name, box, recognized_text)
                if valid_text:
                    assigned_text = valid_text
                    break

        if not assigned_text:
          base_label = class_name
          label_count[base_label] += 1
          unique_label = f"{base_label}_{label_count[base_label]}"
          symbol_ids.append(unique_label)
        else:
          base_label = assigned_text
          symbol_ids.append(base_label)

    for idx, label in enumerate(symbol_ids):
      class_name = class_names[symbol_classes[idx]].lower()
      if "connector" in class_name:
          continue  # skip connectors
      print(label)
    return symbol_ids

def validate_symbol_name(ocr_text, class_name, box, recognized_text):
    """
    Validates and adjusts OCR text for a symbol according to the rules:
    - Relays must contain 'relay'
    - Ground must start with 'G'
    - Motors must contain 'motor'
    - Fuses should include a name and '##A' for amps
    - Connectors use only ID names (no extra text)
    Returns None if text is invalid.
    """
    class_count = defaultdict(int)
    text = ocr_text.replace(" ", "")

    if class_name.lower() == "relay":
        return find_merged_text(text, r'Relay', box, recognized_text)
    elif class_name.lower() == "ground":
        match = re.search(r'G[O0]\d', ocr_text)
        return match.group() if match else None
    elif class_name.lower() == "motor":
        return ocr_text.capitalize() if "motor" in text else None
    elif "fuse" in class_name.lower():
        return find_merged_text(text, r'^\d{2,3}[A]$', box, recognized_text)
        return ocr_text.capitalize() if match else None
    elif "connector" in class_name.lower():
        return None
    else:
        return None


def find_merged_text(text, match_case, symbol_box, recognized_text):
    match = re.match(match_case, text)
    if match:
        text_below = match.group()
        print("Matched:", text_below)

    x1_s, y1_s, x2_s, y2_s = symbol_box
    text_above = ""
    min_distance = float('inf')

    for poly, text in recognized_text:
        px, py, pw, ph = cv2.boundingRect(np.array(poly))
        x1_t, y1_t, x2_t, y2_t = px, py, px + pw, py + ph

        # Check if the text box is above and horizontally overlaps the symbol box
        vertical_overlap = (y2_t >= y1_s - 15) and (y2_t <= y1_s)  # touching/overlapping above

        if vertical_overlap:
            distance = y1_s - y2_t
            if distance < min_distance:
                min_distance = distance
                text_above = text.strip().lower()
                print("Text above found:", text_above)

    if match:
        return f"{text_above}_{text_below}" if text_above else text_below
    else:
        return None

# Removes lines in a text polygon
def remove_lines_overlapping_polygons(lines, polygons):
    def line_intersects_polygon(p1, p2, poly):
        poly_shape = Polygon(poly)
        return poly_shape.contains(Point(p1)) or poly_shape.contains(Point(p2)) or poly_shape.intersects(LineString([p1, p2]))
    return [line for line in lines if not any(line_intersects_polygon((line[0][0], line[0][1]), (line[0][2], line[0][3]), poly) for poly in polygons)]

# Checks if point b is ccw to point c relative to point a
def ccw(a, b, c):
    return (c[1]-a[1]) * (b[0]-a[0]) > (b[1]-a[1]) * (c[0]-a[0])

# Determine if two lines intersect
def do_lines_intersect(a1, a2, b1, b2):
    return ccw(a1, b1, b2) != ccw(a2, b1, b2) and ccw(a1, a2, b1) != ccw(a1, a2, b2)

# Detect lines using Hough transform
def detect_lines(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=threshold, minLineLength=minLineLength, maxLineGap=maxLineGap)
    return lines if lines is not None else []

# Builds a line graph
def build_line_graph(lines):
    G = nx.Graph()
    for line in lines:
        x1, y1, x2, y2 = line[0]
        G.add_edge((x1, y1), (x2, y2))
    return G

# Main method
def detect_connections(input_path, draw_debug_points=True, show_plot=True):
    with open("output.txt", "w") as out_file:
        if os.path.isfile(input_path) and input_path.lower().endswith((".jpg", ".jpeg", ".png")):
            image_files = [input_path]
        elif os.path.isdir(input_path):
            image_files = [os.path.join(input_path, f) for f in sorted(os.listdir(input_path)) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        else:
            print("Invalid input path. Provide an image file or a folder of images.")
            return

        for img_path in image_files:
            filename = os.path.basename(img_path)
            out_file.write(f"Image: {filename}\n")
            image = cv2.imread(img_path)

            # --- 0. Detect Text Boxes (CRAFT) ---
            start = time.time()
            pil_img = Image.open(img_path).convert("RGB")
            polygons = craft_model.get_polygons(pil_img)
            print(f"CRAFT detection: {time.time() - start:.2f}s | "
                  f"{len(polygons)} text regions")

            start = time.time()
            image_cv = cv2.imread(img_path)
            recognized_text = recognize_text_from_polygons(
                image_cv, polygons
            )
            print(f"Tesseract OCR: {time.time() - start:.2f}s | "
                  f"{len(recognized_text)} recognized regions")

            # --- 1. Symbol Detection (YOLO) ---
            start = time.time()
            results = model(img_path)
            symbol_boxes, symbol_classes = [], []

            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = (
                        box.xyxy[0].cpu().numpy().astype(int)
                    )
                    cls_id = int(box.cls[0])
                    if class_names[cls_id] in ignored_classes:
                        continue
                    symbol_boxes.append((x1, y1, x2, y2))
                    symbol_classes.append(cls_id)

            print(f"YOLO detection: {time.time() - start:.2f}s | "
                  f"{len(symbol_boxes)} symbols")

            start = time.time()
            symbol_ids = assign_symbol_ids_with_text(
                symbol_boxes, recognized_text, symbol_classes,
                class_names, margin=50
            )
            print(f"Symbol labeling: {time.time() - start:.2f}s")

            # --- 2. Detect and filter lines ---
            start = time.time()
            wire_lines = remove_lines_overlapping_polygons(
                detect_lines(image), polygons
            )
            print(f"Line detection and polygon filtering: "
                  f"{time.time() - start:.2f}s | "
                  f"{len(wire_lines)} lines")

            # --- 3. Remove short/unconnected wires ---
            start = time.time()
            filtered_lines = []
            for line in wire_lines:
                x1, y1, x2, y2 = line[0]
                p1, p2 = (x1, y1), (x2, y2)
                length = ((x2 - x1)**2 + (y2 - y1)**2)**0.5

                if length < minLineLength:
                    continue

                if any(
                    is_point_near_box(p, box, margin)
                    for box in symbol_boxes
                    for p in [p1, p2]
                ):
                    filtered_lines.append(line)

            wire_lines = filtered_lines
            line_graph = build_line_graph(wire_lines)
            print(f"Wire filtering and graph: "
                  f"{time.time() - start:.2f}s | "
                  f"{len(wire_lines)} lines")

            # --- 4. Map Wire Endpoints to Nearby Symbols ---
            endpoint_to_symbols = {}
            for point in line_graph.nodes:
                for idx, box in enumerate(symbol_boxes):
                    if is_point_near_box(point, box, margin=50):
                        endpoint_to_symbols.setdefault(point, set()).add(idx)

            # --- 5. Build Symbol Connectivity Graph ---
            connections = set()
            for wire_cluster in nx.connected_components(line_graph):
                connected_symbols = set()
                for point in wire_cluster:
                    connected_symbols.update(endpoint_to_symbols.get(point, []))
                if len(connected_symbols) >= 2:
                    names = sorted(set(symbol_ids[i] for i in connected_symbols))
                    for i in range(len(names)):
                        for j in range(i + 1, len(names)):
                            connections.add(tuple(sorted((names[i], names[j]))))

            # --- 6. Visualization ---
            for idx, (x1, y1, x2, y2) in enumerate(symbol_boxes):
                label = symbol_ids[idx]
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                tol_x1, tol_y1, tol_x2, tol_y2 = x1 - margin, y1 - margin, x2 + margin, y2 + margin
                if showSymbolLabel:
                    cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 165, 255), 1)

            for idx, line in enumerate(wire_lines):
                x1, y1, x2, y2 = line[0]
                cv2.line(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
                if showLineLabel:
                    cv2.putText(image, f"line{idx+1}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (59, 71, 110), 1)
                if draw_debug_points:
                    for pt in [(x1, y1), (x2, y2)]:
                        cv2.circle(image, pt, 3, (255, 0, 0), -1)

            for poly in polygons:
                pts = np.array(poly, np.int32).reshape((-1, 1, 2))
                cv2.polylines(image, [pts], isClosed=True, color=(255, 0, 255), thickness=2)

            cv2.imwrite("/content/annotated_output.png", image)
            if show_plot:
              plt.figure(figsize=(12, 8))
              plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
              plt.title(f"{filename} — Labeled Image")
              plt.axis("off")
              plt.show()

            # --- 7. Filter wires not connected to any symbol ---
            connected_wires = [(pt1, pt2) for pt1, pt2 in line_graph.edges if pt1 in endpoint_to_symbols or pt2 in endpoint_to_symbols]
            filtered_graph = nx.Graph()
            filtered_graph.add_edges_from(connected_wires)

            # --- 8. Number Wires ---
            wire_ids = { (pt1, pt2): f"wire_{idx+1}" for idx, (pt1, pt2) in enumerate(filtered_graph.edges) }
            wire_ids.update({(pt2, pt1): label for (pt1, pt2), label in wire_ids.items()})

            # --- 9. Map points to symbols ---
            point_to_symbol = {pt: idx for pt, indices in endpoint_to_symbols.items() for idx in indices}

            # --- 10. Trace all connections ---
            printed = set()
            for component in nx.connected_components(filtered_graph):
                symbols_in_comp = [(pt, point_to_symbol[pt]) for pt in component if pt in point_to_symbol]
                for i in range(len(symbols_in_comp)):
                    for j in range(i + 1, len(symbols_in_comp)):
                        pt1, idx1 = symbols_in_comp[i]
                        pt2, idx2 = symbols_in_comp[j]
                        label1, label2 = symbol_ids[idx1], symbol_ids[idx2]
                        if label1 == label2:
                            continue
                        try:
                            path = nx.shortest_path(filtered_graph, pt1, pt2)
                            steps = [label1] + [wire_ids.get((path[k], path[k+1]), "wire_?") for k in range(len(path)-1)] + [label2]
                            path_str = " → ".join(steps)
                            if (label1, label2) not in printed and (label2, label1) not in printed:
                                print(path_str)
                                out_file.write("  " + path_str + "\n")
                                printed.add((label1, label2))
                        except nx.NetworkXNoPath:
                            continue

# Can specify a folder of images or a single image
# detect_connections("/content/custom_data/images/ac1.png")


import os
import gradio as gr
from PIL import Image

ANNOTATED_PATH = "/content/annotated_output.png"
CONNECTIONS_PATH = os.path.abspath("output.txt")


def analyze_schematic(image_path):
    if image_path is None:
        raise gr.Error("Please upload an electrical schematic image.")

    # Clear stale outputs so a failed run cannot show old results.
    for path in (ANNOTATED_PATH, CONNECTIONS_PATH):
        if os.path.exists(path):
            os.remove(path)

    try:
        # Run the existing AI pipeline without changing its logic.
        detect_connections(image_path, show_plot=False)

        # Check that both expected outputs were generated.
        if not os.path.exists(ANNOTATED_PATH):
            raise gr.Error(
                "The annotated image was not saved. "
                "Check the cv2.imwrite line in detect_connections()."
            )

        if not os.path.exists(CONNECTIONS_PATH):
            raise gr.Error(
                "The connection report was not generated."
            )

        # Read the annotated image in RGB format for display.
        annotated = Image.open(ANNOTATED_PATH).convert("RGB")

        # Read the inferred connections.
        with open(CONNECTIONS_PATH, "r", encoding="utf-8") as f:
            connection_text = f.read()

        return annotated, connection_text, CONNECTIONS_PATH

    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Analysis failed: {e}")


demo = gr.Interface(
    fn=analyze_schematic,
    inputs=gr.Image(
        type="filepath",
        label="Upload an electrical schematic"
    ),
    outputs=[
        gr.Image(label="Annotated schematic"),
        gr.Textbox(
            label="Detected connections",
            lines=15
        ),
        gr.File(label="Download connection report")
    ],
    title="Schematic Labeling using AI",
    description=(
        "Upload an electrical schematic to detect symbols, "
        "text, wires, and inferred connections. "
        "Review the inferred connections against the original diagram."
    ),
    submit_btn="Analyze schematic",
    flagging_mode="never"
)

demo.launch(share=True)

